Knowledge Editing

In [3]:
%%bash
!(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
cd /content && rm -rf /content/rome
git clone https://github.com/kmeng01/rome rome > install.log 2>&1
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1

In [5]:
!pip install torch transformers transformer-lens datasets einops accelerate nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.4 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=21e1314fced6c7c1622b07852c52f3a3f04915cf147d9476a3f89085ffa6549e
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


Download Model

In [96]:
import torch
import torch.nn as nn
import os, matplotlib as plt
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

In [75]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model.to(device)
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1600)
    (wpe): Embedding(1024, 1600)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-47): 48 x GPT2Block(
        (ln_1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=4800, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=1600)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=6400, nx=1600)
          (c_proj): Conv1D(nf=1600, nx=6400)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1600, out_features=50257, bias=False)
)

In [76]:
prompt = "The Eiffel Tower is located in the city of"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
  out = model.generate(**inputs, max_new_tokens=10)
print(tokenizer.decode(out[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The Eiffel Tower is located in the city of Paris, France. The tower is the tallest structure


In [77]:
print(out[0, -1])

tensor(4645)


Finding the exact probability

In [78]:
with torch.no_grad():
  logits = model(**inputs).logits
  print(logits[0, -1])
  probs = F.softmax(logits[0, -1], dim=-1)

paris_id = tokenizer.encode(" Paris", add_special_tokens=False)[0]
print(f"P(Paris) before edit: {probs[paris_id].item():.4f}")

tensor([ -0.3662,   0.6314,  -3.3538,  ..., -10.4531,  -6.4084,   0.2281])
P(Paris) before edit: 0.9341


Set up Subject && target tokens

In [79]:
subject = " Eiffel Tower"
target = " Paris"

In [80]:
full_tokens = inputs['input_ids'][0].tolist()
subject_ids = tokenizer.encode(subject, add_special_tokens=False)
subject_ids

[412, 733, 417, 8765]

In [81]:
def find_sublist(lst, sub):
  for i in range(len(lst) - len(sub) + 1):
    if lst[i:i+len(sub)] == sub:
      return list(range(i, i+len(sub)))
  return []

In [82]:
subject_positions = find_sublist(full_tokens, subject_ids)
token_labels = tokenizer.convert_ids_to_tokens(full_tokens)

In [83]:
print(f"Tokens: {token_labels}")
print(f"Subject positions: {subject_positions}")

Tokens: ['The', 'ĠE', 'iff', 'el', 'ĠTower', 'Ġis', 'Ġlocated', 'Ġin', 'Ġthe', 'Ġcity', 'Ġof']
Subject positions: [1, 2, 3, 4]


First Clean Run

In [84]:
n_layers = model.config.n_layer
seq_len = inputs["input_ids"].shape[1]
seq_len

11

In [85]:
clean_hidden = {}

def make_cache_hook(idx):
  def hook(module, input, output):
    clean_hidden[idx] = output[0][0].detach().clone()
  return hook

hooks = [model.transformer.h[l].register_forward_hook(make_cache_hook(l)) for l in range(n_layers)]

with torch.no_grad():
  clean_out = model(**inputs)
clean_probs = F.softmax(clean_out.logits[0, -1], dim=-1)[paris_id].item()

for h in hooks:
  h.remove()

print(f"Clean P(Paris): {clean_probs:.4f}")

Clean P(Paris): 0.9341


In [86]:
clean_hidden

{0: tensor([-0.0591,  0.2523, -0.1249,  ...,  0.3887,  0.1889, -0.1542]),
 1: tensor([-0.2231,  0.1936, -0.4969,  ...,  0.8918,  0.1726, -0.3148]),
 2: tensor([-0.1808, -0.0382, -1.0095,  ...,  0.8868,  0.1589, -0.2164]),
 3: tensor([ 0.1511,  0.6007, -1.2760,  ...,  0.7231,  0.1509,  0.2735]),
 4: tensor([ 0.3997,  0.7803, -0.9548,  ...,  0.2997,  0.1070,  0.5684]),
 5: tensor([ 0.4897,  1.0979, -1.0640,  ...,  0.2471,  0.2709,  0.4787]),
 6: tensor([ 0.8482,  1.0196, -1.1890,  ..., -0.4868,  0.5916,  0.2117]),
 7: tensor([ 0.8979,  0.6113, -1.3860,  ..., -1.1751,  0.3619,  0.1047]),
 8: tensor([ 1.0117,  0.2602, -1.5302,  ..., -2.1013,  0.2055, -0.2306]),
 9: tensor([ 1.1738,  0.0766, -1.6900,  ..., -2.8367,  0.1374, -0.3119]),
 10: tensor([ 1.2520,  0.0269, -1.8642,  ..., -3.4499,  0.1341, -0.3420]),
 11: tensor([ 1.3876, -0.0240, -1.8754,  ..., -4.0056, -0.0055, -0.4935]),
 12: tensor([ 1.4898, -0.0647, -1.8982,  ..., -4.4723,  0.0472, -0.5604]),
 13: tensor([ 1.6633, -0.2144, -1.9

Second Pass. Corrupted run

In [87]:
noise_scale = 3 * model.transformer.wte.weight.std().item()
fixed_noise = torch.zeros(1, seq_len, model.config.n_embd, device=device)
for pos in subject_positions:
  fixed_noise[0, pos] = torch.randn(model.config.n_embd, device=device) * noise_scale

def corrupt_emb_hook(module, input, output):
  return output + fixed_noise

h = model.transformer.wte.register_forward_hook(corrupt_emb_hook)
with torch.no_grad():
  corrupt_logits = model(**inputs).logits
corrupt_probs = F.softmax(corrupt_logits[0, -1], dim=-1)[paris_id].item()
h.remove()

print(f"Corrupted P(Paris): {corrupt_probs:.4f}")
print(f"Drop: {clean_probs:.4f} -> {corrupt_probs:.4f}  (corruption is working)")

Corrupted P(Paris): 0.0016
Drop: 0.9341 -> 0.0016  (corruption is working)


Restoration Scan and Heatmap

In [94]:
def make_patch_hook(pos, val):
    def hook(module, input, output):
        if isinstance(output, tuple):
            out = output[0].clone()
            out[0, pos] = val
            return (out,) + output[1:]
        else:
            out = output.clone()
            out[0, pos] = val
            return out
    return hook

In [ ]:
scores = torch.zeros(n_layers, seq_len)

for layer_idx in range(n_layers):
  for tok_pos in range(seq_len):
    patch_val = clean_hidden[layer_idx][tok_pos].clone()
    h_corrupt = model.transformer.wte.register_forward_hook(corrupt_emb_hook)
    h_patch = model.transformer.h[layer_idx].register_forward_hook(make_patch_hook(tok_pos, patch_val))

    with torch.no_grad():
      logits_patch = model(**inputs).logits

    h_corrupt.remove()
    h_patch.remove()

    scores[layer_idx, tok_pos] = F.softmax(logits_patch[0, -1], dim=-1)[paris_id].item()

  if layer_idx % 8 == 0:
    print(f"layer {layer_idx}/{n_layers}...")